# Twilight Imperium Fandom Data Processor
## Scrape Fandom Wiki and Add to Vector Database

This notebook scrapes additional game data from the Twilight Imperium Fandom wiki (relics, action cards, etc.) and adds it to your existing vector database.


In [1]:
# Import necessary libraries
import json
import os
from pathlib import Path
from typing import List, Dict, Any

# Import the Fandom scraper
from fandom_data_scraper import TwilightFandomDataScraper

# LangChain imports
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.schema import Document as LangChainDocument

# Load environment variables
from dotenv import load_dotenv
load_dotenv(override=True)

print("✅ All libraries imported successfully")


✅ All libraries imported successfully


## Step 1: Scrape Fandom Data
Scrape additional game data from the Fandom wiki


In [2]:
# Initialize the Fandom scraper
scraper = TwilightFandomDataScraper(debug_mode=False)

# Choose what to scrape (you can customize this list)
data_types_to_scrape = [
    "relics",
    "action_cards",
    "agenda_cards",
    "technologies",
    "planets",
    "objectives"
]

print(f"📋 Will scrape {len(data_types_to_scrape)} data types:")
for dt in data_types_to_scrape:
    print(f"  • {dt}")


✅ Fandom data scraper initialized
📁 Data will be saved to: processed_rules\fandom_data
📋 Will scrape 6 data types:
  • relics
  • action_cards
  • agenda_cards
  • technologies
  • planets
  • objectives


In [3]:
# Scrape the data
print("🚀 Starting Fandom scraping...")
print("⏳ This may take a few minutes (2 second delay between requests)...\n")

fandom_data = scraper.scrape_all_data_types(data_types_to_scrape)

# Save the scraped data
scraper.save_data(fandom_data)


🚀 Starting Fandom scraping...
⏳ This may take a few minutes (2 second delay between requests)...

🚀 Starting to scrape 6 data types...

🔍 Scraping relics...
📄 Description: Powerful artifacts found on frontier planets
  ❌ Error scraping relics: 404 Client Error: Not Found for url: https://twilight-imperium.fandom.com/wiki/Relic

🔍 Scraping action_cards...
📄 Description: Cards players can play during tactical or strategic actions
  ❌ Error scraping action_cards: 404 Client Error: Not Found for url: https://twilight-imperium.fandom.com/wiki/Action_Card

🔍 Scraping agenda_cards...
📄 Description: Political cards revealed during the Agenda Phase
  ❌ Error scraping agenda_cards: 404 Client Error: Not Found for url: https://twilight-imperium.fandom.com/wiki/Agenda_Card

🔍 Scraping technologies...
📄 Description: Upgrades and advancements for factions
  ✅ Successfully scraped technologies
  📊 Main text: 8926 characters
  📊 Sections: 11
  📊 Tables: 11
  📊 Lists: 5

🔍 Scraping planets...
📄 Descrip

## Step 2: Convert to Text and Chunk
Convert the scraped data to text format and chunk it for embedding


In [4]:
# Convert each data type to text
all_fandom_text = []

for data_type, result in fandom_data["data_types"].items():
    if result.get("scraped_successfully"):
        text = scraper.convert_to_text_for_embedding(result)
        if text:
            all_fandom_text.append({
                "data_type": data_type,
                "text": text,
                "source_url": result.get("source_url", "")
            })
            print(f"✅ Converted {data_type}: {len(text)} characters")

print(f"\n📊 Total data sources ready: {len(all_fandom_text)}")


✅ Converted technologies: 13381 characters

📊 Total data sources ready: 1


In [5]:
# Configure text splitter (same settings as existing pipeline)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", ", ", " ", ""],
    keep_separator=True
)

print("🔧 Text splitter configured with consistent settings")


🔧 Text splitter configured with consistent settings


In [6]:
# Chunk all the Fandom data
all_fandom_chunks = []

for item in all_fandom_text:
    data_type = item["data_type"]
    text = item["text"]
    
    print(f"\n📝 Chunking {data_type}...")
    
    # Split into chunks
    chunks = text_splitter.split_text(text)
    
    # Create chunks with metadata
    for i, chunk in enumerate(chunks):
        chunk_metadata = {
            'source': f'fandom_{data_type}',
            'doc_type': f'Fandom Wiki - {data_type.replace("_", " ").title()}',
            'chunk_id': f'fandom_{data_type}_chunk_{i:03d}',
            'chunk_index': i,
            'total_chunks': len(chunks),
            'char_count': len(chunk),
            'word_count': len(chunk.split()),
            'source_url': item.get('source_url', '')
        }
        
        all_fandom_chunks.append({
            'content': chunk.strip(),
            'metadata': chunk_metadata
        })
    
    print(f"  ✅ Created {len(chunks)} chunks")

print(f"\n📊 Total chunks from Fandom: {len(all_fandom_chunks)}")



📝 Chunking technologies...
  ✅ Created 21 chunks

📊 Total chunks from Fandom: 21


## Step 2b: Load Codex sections (Relics, Action Cards, Faction Technologies)
If available, pull sections parsed from the Codex page and include them in chunking.


In [7]:
# Load codex sections if present
from pathlib import Path
import json

codex_file = Path("processed_rules/fandom_data/codex_sections.json")
codex_sections = []
if codex_file.exists():
    with open(codex_file, 'r', encoding='utf-8') as f:
        codex_sections = json.load(f)
    print(f"✅ Loaded {len(codex_sections)} sections from Codex")
else:
    print("ℹ️ No codex_sections.json found; skipping Codex ingestion")

# Convert Codex sections into text items for chunking
for sec in codex_sections:
    data_type = "codex_section"
    section_label = sec.get("section_line", "Codex Section")
    text = f"=== {section_label} (from {sec.get('page')}) ===\n\n" + (sec.get("text") or "")
    all_fandom_text.append({
        "data_type": data_type,
        "text": text,
        "source_url": "https://twilight-imperium.fandom.com/wiki/Codex_I_-_IV_(Fourth_Edition_Expansion)",
    })
print(f"📎 Total data sources after Codex: {len(all_fandom_text)}")


✅ Loaded 4 sections from Codex
📎 Total data sources after Codex: 5


## Step 3: Add to Existing Vector Store
Load the existing vector database and add the new Fandom chunks


In [8]:
# Verify OpenAI API key
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("❌ OpenAI API key not found!")
elif api_key.startswith("sk-"):
    print("✅ OpenAI API key found")
    print(f"Key starts with: {api_key[:20]}...")
else:
    print("⚠️  API key format looks incorrect")

# Initialize embedding model (same as existing pipeline)
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=api_key
)
print("✅ Embeddings model initialized")


✅ OpenAI API key found
Key starts with: sk-proj-gU0tE3P-TebB...
✅ Embeddings model initialized


In [10]:
# Load existing vector store
processed_rules_dir = Path("processed_rules")
vector_store_path = processed_rules_dir / "vector_store"

if not vector_store_path.exists():
    print("❌ Error: Existing vector store not found!")
    print("Please run embedding_generator.ipynb first")
else:
    print(f"📂 Loading existing vector store from: {vector_store_path}")
    
    vector_store = FAISS.load_local(
        str(vector_store_path),
        embeddings_model,
        allow_dangerous_deserialization=True
    )
    
    print(f"✅ Loaded existing vector store")
    print(f"📊 Current vectors in store: {vector_store.index.ntotal}")


📂 Loading existing vector store from: processed_rules\vector_store
✅ Loaded existing vector store
📊 Current vectors in store: 652


In [11]:
# Convert chunks to LangChain Documents
print("📄 Converting Fandom chunks to LangChain Documents...")

fandom_documents = []
for chunk in all_fandom_chunks:
    doc = LangChainDocument(
        page_content=chunk['content'],
        metadata=chunk['metadata']
    )
    fandom_documents.append(doc)

print(f"✅ Created {len(fandom_documents)} Document objects")

# Preview a sample
if fandom_documents:
    sample = fandom_documents[0]
    print(f"\n🔍 Sample Document:")
    print(f"  Source: {sample.metadata['source']}")
    print(f"  Doc Type: {sample.metadata['doc_type']}")
    print(f"  Preview: {sample.page_content[:150]}...")


📄 Converting Fandom chunks to LangChain Documents...
✅ Created 21 Document objects

🔍 Sample Document:
  Source: fandom_technologies
  Doc Type: Fandom Wiki - Technologies
  Preview: === TECHNOLOGIES ===

Rules

Refence for Base Game tech.

Players obtain technology in the form of technology cards.

Refence for POK tech.

Technolog...


In [12]:
# Add Fandom documents to the existing vector store
print("🔥 Adding Fandom documents to vector store...")
print(f"📊 Current vectors: {vector_store.index.ntotal}")
print(f"➕ Adding {len(fandom_documents)} new vectors")
print("⏳ Generating embeddings (this may take a few minutes)...")

try:
    # Add documents to vector store
    vector_store.add_documents(fandom_documents)
    
    print(f"✅ Successfully added Fandom documents!")
    print(f"📈 Total vectors now: {vector_store.index.ntotal}")
    
except Exception as e:
    print(f"❌ Error adding documents: {e}")


🔥 Adding Fandom documents to vector store...
📊 Current vectors: 652
➕ Adding 21 new vectors
⏳ Generating embeddings (this may take a few minutes)...
✅ Successfully added Fandom documents!
📈 Total vectors now: 673


In [13]:
# Test the updated vector store
print("🧪 Testing updated vector store with Fandom-related queries...")

test_queries = [
    "What are relics?",
    "Tell me about action cards",
    "What technologies are available?",
    "How do objectives work?"
]

print("\n🔍 Sample Query Results:")
print("="*50)

for query in test_queries[:2]:
    print(f"\n🔸 Query: '{query}'")
    
    try:
        results = vector_store.similarity_search(query, k=3)
        
        print(f"   Found {len(results)} results:")
        for i, doc in enumerate(results[:2], 1):
            print(f"   📄 Result {i}:")
            print(f"      Source: {doc.metadata['source']}")
            print(f"      Doc Type: {doc.metadata.get('doc_type', 'N/A')}")
            print(f"      Preview: {doc.page_content[:120]}...")
            print()
    except Exception as e:
        print(f"   ❌ Error: {e}")

print("✅ Testing complete!")


🧪 Testing updated vector store with Fandom-related queries...

🔍 Sample Query Results:

🔸 Query: 'What are relics?'
   Found 3 results:
   📄 Result 1:
      Source: twilight_pravila
      Doc Type: Twilight Pravila (Rules)
      Preview: Q: What happens to your relics and relic fragments when you're eliminated?

A: Relics are purged, relic fragments discar...

   📄 Result 2:
      Source: rulebook
      Doc Type: Official Rulebook
      Preview: . Then, they place the matching attachment token on that planet on the game board. That planet is modified by the explor...


🔸 Query: 'Tell me about action cards'
   Found 3 results:
   📄 Result 1:
      Source: learn_to_play
      Doc Type: Learn to Play Guide
      Preview: . Action Card Deck Agenda Deck Objective Decks 14 ARCHON TAU 1 1 ARCHON REN 3 2 The was nox dus kill N has fore min STAR...

   📄 Result 2:
      Source: learn_to_play
      Doc Type: Learn to Play Guide
      Preview: . Many action cards, faction sheets, and even some te

In [14]:
# Save the updated vector store
print(f"💾 Saving updated vector store to: {vector_store_path}")

try:
    # Save the FAISS vector store
    vector_store.save_local(str(vector_store_path))
    print("✅ Updated vector store saved successfully!")
    
    # Update embedding configuration
    config_path = processed_rules_dir / "embedding_config.json"
    
    with open(config_path, 'r', encoding='utf-8') as f:
        embedding_config = json.load(f)
    
    # Update config with Fandom data info
    embedding_config['total_vectors'] = vector_store.index.ntotal
    embedding_config['fandom_documents_added'] = len(all_fandom_chunks)
    
    # Add info about each Fandom data type
    if 'sources' not in embedding_config:
        embedding_config['sources'] = {}
    
    for item in all_fandom_text:
        data_type = item['data_type']
        num_chunks = len([c for c in all_fandom_chunks if c['metadata']['source'] == f'fandom_{data_type}'])
        embedding_config['sources'][f'fandom_{data_type}_chunks'] = num_chunks
    
    # Save updated config
    with open(config_path, 'w', encoding='utf-8') as f:
        json.dump(embedding_config, f, indent=2, ensure_ascii=False)
    
    print(f"⚙️  Updated configuration saved to: {config_path}")
    
except Exception as e:
    print(f"❌ Error saving: {e}")

print(f"\n🎉 Fandom Data Integration Complete!")
print(f"📊 Summary:")
print(f"  - Scraped {len(data_types_to_scrape)} data types from Fandom")
print(f"  - Created {len(all_fandom_chunks)} chunks")
print(f"  - Total vectors in database: {vector_store.index.ntotal}")
print(f"\n🚀 Your chatbot now knows about:")
for dt in data_types_to_scrape:
    print(f"  ✅ {dt.replace('_', ' ').title()}")
print(f"\n💡 Try asking questions like:")
print(f"  • 'What are the different relics?'")
print(f"  • 'How do action cards work?'")
print(f"  • 'Tell me about technology in the game'")


💾 Saving updated vector store to: processed_rules\vector_store
✅ Updated vector store saved successfully!
⚙️  Updated configuration saved to: processed_rules\embedding_config.json

🎉 Fandom Data Integration Complete!
📊 Summary:
  - Scraped 6 data types from Fandom
  - Created 21 chunks
  - Total vectors in database: 673

🚀 Your chatbot now knows about:
  ✅ Relics
  ✅ Action Cards
  ✅ Agenda Cards
  ✅ Technologies
  ✅ Planets
  ✅ Objectives

💡 Try asking questions like:
  • 'What are the different relics?'
  • 'How do action cards work?'
  • 'Tell me about technology in the game'
